In [ ]:
import hashlib
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import harmony

harmony_host_url = "https://harmony.uat.earthdata.nasa.gov"

In [ ]:
environments = {
    "http://localhost:3000": harmony.Environment.LOCAL,
    "https://harmony.sit.earthdata.nasa.gov": harmony.Environment.SIT,
    "https://harmony.uat.earthdata.nasa.gov": harmony.Environment.UAT,
    "https://harmony.earthdata.nasa.gov": harmony.Environment.PROD,
}
assert harmony_host_url in environments

harmony_client = harmony.Client(env=environments[harmony_host_url])
request = harmony.Request(
    collection=harmony.Collection(id="C1273831195-ASF"),
    granule_name=[
        "NISAR_L2_PR_GCOV_015_156_A_010_2005_DVDV_A_20230619T000803_20230619T000818_T05000_N_P_J_001"
    ],
    format="image/png",
    labels=["nisar-py-rtests"],
)
job_id = harmony_client.submit(request)
harmony_client.wait_for_processing(job_id)

with open("expected_md5sums.json") as f:
    expected_md5sums = json.load(f)

with TemporaryDirectory() as temp_dir:
    urls = list(harmony_client.result_urls(job_id))

    # Verify that there are the expected number of files for each filetype.
    assert len(urls) == 109
    assert len([url for url in urls if url.endswith(".txt")]) == 1
    assert len([url for url in urls if url.endswith(".png")]) == 36
    assert len([url for url in urls if url.endswith(".pgw")]) == 36
    assert len([url for url in urls if url.endswith(".png.aux.xml")]) == 36

    # Verify that the output file checksums match the expected checksums.
    output_files = [
        Path(harmony_client.download(url, temp_dir).result())
        for url in urls
        if not url.endswith(".txt")
    ]
    for file in output_files:
        key = file.name.split(".", maxsplit=1)[1]
        assert (
            hashlib.md5(file.read_bytes()).hexdigest() == expected_md5sums[key]
        ), f"md5sum for {key} does not match expected value"